# SigLIP + hai nguồn OCR trên keyframe của BTC → Volume `keyframes-btc`

`docs/triet/ke-hoach-offline-ingest.md` §2.1, §2.3, §7 · config `config/ingest_shards_batch1_btc_images.yaml`

Chạy **sau** `02-keyframes-btc-import.ipynb`. Cùng bundle, cùng volume, **cùng cách chia người** —
đổi `MEMBER` ở ô đầu là ra đúng phần việc bạn đã nhập ở lượt trước.

### Ba tầng, và tầng nào KHÔNG có ở đây

| Tầng | Model | Ra artifact |
|---|---|---|
| `n4_siglip` | `google/siglip-so400m-patch14-384` (tháp ảnh) | `siglip_vectors` |
| `n5_ocr_easyocr` | easyocr vi+en | `ocr_text` |
| `n5_ocr_qwen` | `Qwen/Qwen2-VL-7B-Instruct` 4-bit | `sviptr_ocr` |

`n4_objects` **không có**: BTC phát hành sẵn nhãn object và notebook 02 đã nạp chúng. Chạy
detector của ta lên chính bộ keyframe này là phép đo khác (§3.6) và là việc khác.

`n6_recap` / `n6_bge_m3` **không có**: Lead đã hoãn tầng ReCap.

### Vì sao ảnh tải lại từ zip chứ không kéo từ Volume

Volume ở đường BTC là **ghi một chiều**. Kéo ảnh về là ~177.000 lần `read_file`; tải lại gói
`Keyframes_*.zip` là 1,4–5,7 GB một lần, có resume. Cùng dữ liệu, khác hai bậc chi phí.

### Ba model cùng nằm trong VRAM một lúc — có chủ ý

SigLIP ~1,1 GB + easyocr ~0,1 GB + Qwen2-VL 4-bit ~5,5 GB ≈ **6,7 GB / 15 GB của T4**. Giữ cả ba
để mỗi model chỉ nạp **một lần cho cả phiên**; nạp lại theo từng video tốn **5–16 giờ mỗi người
chỉ để nạp** (đo trên số video thật của từng người).

> Nếu gặp OOM: đặt `KEEP_MODELS = False` ở ô đầu. Notebook sẽ thả model sau mỗi video — đúng hành
> vi cũ của §4.2, đổi lại là thời gian nạp.


## Dùng lại private Kaggle cache

Attach cùng Dataset đã dùng cho notebook 02. Kaggle thường tự giải nén archive thành thư mục;
notebook nhận cả layout đã giải nén lẫn ZIP nguyên bản trong `/kaggle/input`, giữ cache read-only
và chỉ tải từ website khi cả hai dạng đều thiếu. Không copy 15 GiB sang `/kaggle/working`.


## Step 0a — repo + gói

In [ ]:
MEMBER      = "trong"          # <-- ĐỔI THÀNH TÊN BẠN
SHARDS      = "config/ingest_shards_batch1_btc_images.yaml"
TRANSPORT   = "modal"
VOLUME      = "keyframes-btc"  # CÙNG volume với notebook 02
VOL_VERSION = 2
KEEP_MODELS = True             # False = thả model sau mỗi video (xem ô mô tả trên)
REPO_URL    = "https://github.com/AIVIETNAM-AIO-Triet-Descartes/multimodal-fusion-engine.git"
REPO_REF    = "feature/trong/online-retrieval"

import os, subprocess, sys
from pathlib import Path

REPO = Path("/kaggle/input/mfe-repo")
if not REPO.is_dir():
    REPO = Path("/kaggle/working/mfe")
    if not REPO.is_dir():
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        url = REPO_URL.replace("https://", f"https://{token}@")
        subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_REF, url, str(REPO)],
                       check=True, capture_output=True)   # token không lọt vào log
sys.path.insert(0, str(REPO / "src"))

# Bản .lock thắng bản dải phiên bản — 5 người phải cùng phiên bản (§2.5).
req = REPO / "notebooks" / "requirements-kaggle.lock.txt"
if not req.is_file():
    req = REPO / "notebooks" / "requirements-kaggle-btc-images.txt"
print(f"cài từ {req.name}")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=True)

## Step 0b — Modal, phân công, việc đã xong

In [ ]:
import logging, yaml
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

from kaggle_secrets import UserSecretsClient
from mfe.offline.bundle import BundleLayout, BundleWriter, StageStatus, load_shard_plan
from mfe.offline.stages import STAGES, StageContext, read_keyframes, unload_models
from mfe.offline.transport import build_transport

secrets = UserSecretsClient()
os.environ["MODAL_TOKEN_ID"] = secrets.get_secret("MODAL_TOKEN_ID")
os.environ["MODAL_TOKEN_SECRET"] = secrets.get_secret("MODAL_TOKEN_SECRET")

RAW       = yaml.safe_load((REPO / SHARDS).read_text(encoding="utf-8"))
plan      = load_shard_plan(REPO / SHARDS)
layout    = BundleLayout(plan.bundle)
transport = build_transport(TRANSPORT, volume=VOLUME,
                            local_root="/kaggle/working/bundle_local",
                            version=VOL_VERSION)
WORK   = Path("/kaggle/working/scratch"); WORK.mkdir(parents=True, exist_ok=True)
ZIPS   = Path("/kaggle/working/zips");    ZIPS.mkdir(parents=True, exist_ok=True)
KAGGLE_INPUT = Path("/kaggle/input")       # private Kaggle Datasets, read-only
writer = BundleWriter(transport, layout, MEMBER, WORK)

assert plan.keyframe_set == "btc"
STAGE_LIST = plan.stages
PARAMS     = RAW.get("params", {})
EXTRAS     = RAW["extras"]

MY_ZIPS   = plan.zips_for(MEMBER)
MY_VIDEOS = plan.videos_for(MEMBER)
MY_VIDEOS = MY_VIDEOS[:1]
status    = StageStatus.load(transport, layout, MEMBER)

# Điều kiện tiên quyết: notebook 02 phải xong TRƯỚC. Thiếu `n0_import_btc` thì
# `map-keyframes` của video đó chưa lên Volume, và bundle sẽ có vector cho một
# bộ keyframe mà chính nó không mô tả.
missing_import = [v for v in MY_VIDEOS if not status.is_done("n0_import_btc", v)]
todo = [(s, v) for v in MY_VIDEOS for s in STAGE_LIST if not status.is_done(s, v)]

print(f"bundle  : {plan.bundle}  ->  volume {VOLUME} (v{VOL_VERSION})")
print(f"người   : {MEMBER}")
print(f"zip     : {', '.join(n for n, _ in MY_ZIPS)}")
print(f"video   : {len(MY_VIDEOS)} (tổng {len(plan.videos)})")
print(f"tầng    : {', '.join(STAGE_LIST)}")
print(f"còn lại : {len(todo)} việc / {len(MY_VIDEOS) * len(STAGE_LIST)}")
if missing_import:
    print()
    print(f"CHU Y: {len(missing_import)} video CHUA nhap bang notebook 02: "
          f"{', '.join(missing_import[:6])}{' …' if len(missing_import) > 6 else ''}")
    print("   Chay xong 02 truoc — notebook nay se BO QUA chung.")

In [ ]:
import shutil, zipfile

def free_gb():
    return shutil.disk_usage("/kaggle/working").free / 2**30

def check_size(target, expect):
    """Kích thuoc dung hay khong. `expect=None` = khong biet, bo qua.

    Vi sao doc rieng chu khong tin `wget` bao thanh cong: `-c` noi lai bang HTTP
    Range, va **neu may chu lo Range** thi no tra ve ca file voi ma 200, con
    wget noi phan do vao sau phan da co. File ra to hon that, hong tu giua, va
    khong co canh bao nao. Kiem kich thuoc la thu duy nhat thay duoc dieu do.
    Cat cut (dut giua chung) thi cung bat duoc luon.
    """
    if expect is None:
        return
    actual = target.stat().st_size
    if actual != expect:
        target.unlink(missing_ok=True)
        raise RuntimeError(
            f"{target.name}: tai ve {actual:,} byte, cho doi {expect:,} "
            f"(lech {actual - expect:+,}). Da xoa file hong — chay lai o nay."
        )


def validate_zip(path):
    """Đọc central directory để bắt ZIP rỗng, cắt cụt hoặc không hợp lệ."""
    try:
        with zipfile.ZipFile(path) as archive:
            if not archive.infolist():
                raise RuntimeError(f"{path}: ZIP rỗng")
    except zipfile.BadZipFile as err:
        raise RuntimeError(f"{path}: không phải ZIP hoàn chỉnh") from err


def find_kaggle_zip(name, expect_bytes=None):
    """Tìm một ZIP nguyên bản trong private Kaggle Dataset đã attach.

    Cache trong `/kaggle/input` là read-only: chỉ đọc, không copy và không xóa.
    Nếu có nhiều file cùng tên hoặc sai kích thước thì dừng thay vì chọn nhầm.
    """
    if not KAGGLE_INPUT.is_dir():
        return None
    candidates = sorted(p for p in KAGGLE_INPUT.rglob(name) if p.is_file())
    if not candidates:
        return None
    if len(candidates) != 1:
        found = ", ".join(str(p) for p in candidates)
        raise RuntimeError(f"{name}: thấy nhiều bản trong /kaggle/input: {found}")
    cached = candidates[0]
    actual = cached.stat().st_size
    if expect_bytes is not None and actual != expect_bytes:
        raise RuntimeError(
            f"{cached}: {actual:,} byte, chờ đợi {expect_bytes:,}; "
            "hãy upload lại đúng ZIP nguyên bản"
        )
    validate_zip(cached)
    print(f"    cache Kaggle: {cached} ({actual / 2**30:.2f} GB)")
    return cached


def download(url, name, expect_bytes=None):
    """Ưu tiên Kaggle Dataset; thiếu cache mới tải, có resume."""
    cached = find_kaggle_zip(name, expect_bytes)
    if cached is not None:
        return cached
    target = ZIPS / name
    if target.is_file():
        size = target.stat().st_size
        complete = expect_bytes is not None and size == expect_bytes
        if expect_bytes is None:
            try:
                validate_zip(target)
                complete = True
            except RuntimeError:
                complete = False
        if complete:
            validate_zip(target)
            print(f"    {name}: đã có ({size / 2**30:.2f} GB)")
            return target
        # File dang do tu phien truoc — `wget -c` chay tiep, khong tai lai.
        print(f"    {name}: đang dở {size / 2**30:.2f}/{expect_bytes / 2**30:.2f} GB, chạy tiếp")
    print(f"    tải {name} … (đĩa trống {free_gb():.1f} GB)")
    # --read-timeout la thu quan trong nhat o day. Khong co no, mot ket noi bi
    # bop xuong ~400 B/s van la ket noi "song": wget ngoi doi vo han thay vi noi
    # lai. Co no thi 60 giay khong nhan duoc byte nao la ngat va thu lai, va lan
    # noi lai thuong duoc toc do binh thuong.
    subprocess.run(
        ["wget", "-q", "--show-progress", "-c",
         "--timeout=30", "--read-timeout=60", "--tries=0", "--waitretry=10",
         "--retry-connrefused", "-O", str(target), url],
        check=True,
    )
    check_size(target, expect_bytes)
    try:
        validate_zip(target)
    except RuntimeError as err:
        target.unlink(missing_ok=True)
        raise RuntimeError(
            f"{name}: tải xong nhưng ZIP hỏng; đã xóa, hãy chạy lại"
        ) from err
    return target


def index_kaggle_extracted_archives(names):
    """Index archive directories that Kaggle unpacked under `/kaggle/input`."""
    targets = {Path(name).stem for name in names}
    found = {name: [] for name in targets}
    if not KAGGLE_INPUT.is_dir():
        return {}
    for root, directories, _files in os.walk(KAGGLE_INPUT):
        keep = []
        for directory in directories:
            if directory in targets:
                found[directory].append(Path(root) / directory)
            else:
                keep.append(directory)
        # Do not descend into a matched archive: it may contain ~90k images.
        directories[:] = keep
    indexed = {}
    for name, paths in found.items():
        if len(paths) > 1:
            locations = ', '.join(str(path) for path in paths)
            raise RuntimeError(f"{name}: thấy nhiều cache đã giải nén: {locations}")
        if paths:
            indexed[name] = paths[0]
    return indexed


archive_names = [name for name, _url in MY_ZIPS] + ["map-keyframes-aic25-b1.zip"]
EXTRACTED_ARCHIVES = index_kaggle_extracted_archives(archive_names)


def find_kaggle_extracted_dir(archive_name, child):
    """Return one validated read-only child of an unpacked Kaggle archive."""
    archive = EXTRACTED_ARCHIVES.get(Path(archive_name).stem)
    if archive is None:
        return None
    extracted = archive / child
    if not extracted.is_dir():
        raise RuntimeError(f"{archive}: thiếu thư mục {child}")
    print(f"    cache Kaggle đã giải nén: {extracted}")
    return extracted

def extract_video(zip_path, video_id, into):
    """Bung ảnh của ĐÚNG một video. Bung cả gói L25 là 5,67 GB nằm chờ vô ích."""
    out = into / video_id
    out.mkdir(parents=True, exist_ok=True)
    count = 0
    with zipfile.ZipFile(zip_path) as archive:
        for name in archive.namelist():
            parts = Path(name).parts
            if len(parts) < 2 or parts[-2] != video_id or name.endswith("/"):
                continue
            with archive.open(name) as src, (out / parts[-1]).open("wb") as dst:
                shutil.copyfileobj(src, dst, length=1 << 20)
            count += 1
    return out, count

def extract_one_file(zip_path, inner_suffix, dest):
    with zipfile.ZipFile(zip_path) as archive:
        match = [n for n in archive.namelist() if n.endswith(inner_suffix)]
        if not match:
            raise FileNotFoundError(f"{inner_suffix} khong co trong {zip_path.name}")
        dest.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(match[0]) as src, dest.open("wb") as dst:
            shutil.copyfileobj(src, dst)
    return dest

# `map-keyframes` lấy từ gói ~1 MB của BTC, KHÔNG kéo từ Volume: cùng một file,
# mà đường này không tốn lượt đọc Volume nào.
print("gói dùng chung:")
MAPS_DIR = find_kaggle_extracted_dir("map-keyframes-aic25-b1.zip", "map-keyframes")
MAPS_ZIP = None if MAPS_DIR is not None else download(
    EXTRAS["map_keyframes"], "map-keyframes-aic25-b1.zip"
)
print(f"đĩa trống: {free_gb():.1f} GB")

## Step 1 — vòng lặp

Theo **video**, không theo tầng: model đã nằm trong cache nên số lần nạp giống hệt cách chia theo
tầng, nhưng mỗi video chỉ cần ~35 MB ảnh trên đĩa thay vì cả gói đã bung.

Marker cắm theo từng *(tầng, video)*. Phiên đứt giữa chừng thì lần chạy sau bỏ đúng những việc đã
xong, kể cả khi nó đứt giữa hai tầng của cùng một video.

`merge=True` khi ghi manifest: bundle này được đổ bởi **ba notebook** (02 nhập, 03 đây, 04 video)
và cả ba cùng ghi `part-manifest/{video}.json`. Không gộp thì mỗi lượt xoá sạch artifact của lượt
trước, còn marker vẫn báo "xong" nên không lần chạy nào phục hồi lại.

In [ ]:
import time, traceback

failures = []
started = time.perf_counter()
done_count = 0

def run_one(video_id, kf_dir, map_csv, stages):
    """Chạy các tầng còn thiếu của MỘT video. Trả về (keyframes, markers)."""
    keyframes = read_keyframes(map_csv, kf_dir)
    ctx = StageContext(video_id, Path("/dev/null"), keyframes, writer, layout, WORK)
    markers = []
    for stage in stages:
        ctx.params = dict(PARAMS.get(stage, {}))
        marker = STAGES[stage](ctx)
        markers.append(marker)
        status.done.add((stage, video_id))
        print(f"      {stage:<16} {marker.get('rows', '-'):>6} dòng  {marker.get('elapsed_s')}s")
    return keyframes, markers

for zip_name, zip_url in MY_ZIPS:
    mine = set(MY_VIDEOS)
    videos = [v for v in plan.videos_in(zip_name)
              if v in mine and status.is_done("n0_import_btc", v)]
    pending = [v for v in videos if any(not status.is_done(s, v) for s in STAGE_LIST)]
    if not pending:
        print()
        print(f"=== {zip_name}: xong hết, không tải ===")
        continue

    print()
    print(f"=== {zip_name}: còn {len(pending)}/{len(videos)} video ===")
    kf_root = find_kaggle_extracted_dir(zip_name, "keyframes")
    kf_zip = None if kf_root is not None else download(
        zip_url, zip_name, plan.zip_bytes.get(zip_name)
    )
    zip_started, zip_done = time.perf_counter(), 0

    for video_id in pending:
        stages = [s for s in STAGE_LIST if not status.is_done(s, video_id)]
        try:
            if kf_root is not None:
                kf_dir = kf_root / video_id
                if not kf_dir.is_dir():
                    raise FileNotFoundError(f"{video_id}: thiếu trong {kf_root}")
                n_img = sum(
                    1 for path in kf_dir.iterdir()
                    if path.is_file() and path.suffix.lower() == ".jpg"
                )
            else:
                kf_dir, n_img = extract_video(kf_zip, video_id, WORK / "keyframes")
            if MAPS_DIR is not None:
                map_csv = MAPS_DIR / f"{video_id}.csv"
                if not map_csv.is_file():
                    raise FileNotFoundError(f"{map_csv}: không tồn tại")
            else:
                map_csv = extract_one_file(MAPS_ZIP, f"/{video_id}.csv",
                                           WORK / "map-keyframes" / f"{video_id}.csv")
            print(f"  {video_id}  {n_img} ảnh  ({len(stages)} tầng)")
            keyframes, markers = run_one(video_id, kf_dir, map_csv, stages)
            writer.write_part_manifest(video_id, markers, keyframes=len(keyframes), merge=True)
            done_count += 1
            zip_done += 1
            left = (len(pending) - zip_done) * (time.perf_counter() - zip_started) / zip_done
            print(f"  [{done_count:>3}] {video_id} xong | còn ~{left / 3600:.1f}h cho gói này")
        except Exception as err:
            failures.append((video_id, repr(err)))
            print(f"  [LỖI] {video_id}: {err}")
            traceback.print_exc(limit=3)
        finally:
            shutil.rmtree(WORK / "keyframes" / video_id, ignore_errors=True)
            (WORK / "map-keyframes" / f"{video_id}.csv").unlink(missing_ok=True)
            if not KEEP_MODELS:
                unload_models()

    if kf_zip is None:
        print(f"    giữ cache Kaggle đã giải nén: {kf_root}")
    elif kf_zip.parent == ZIPS:
        kf_zip.unlink(missing_ok=True)
        print(f"    đã xoá {zip_name}, đĩa trống {free_gb():.1f} GB")
    else:
        print(f"    giữ ZIP cache chỉ-đọc: {kf_zip}")

print()
print("=" * 72)
print(f"phiên này: {done_count} video, {len(failures)} lỗi, "
      f"{(time.perf_counter() - started) / 3600:.1f}h")
for video_id, err in failures:
    print(f"  {video_id}: {err}")

## Step 2 — chốt phiên

In [ ]:
unload_models()

fresh = StageStatus.load(transport, layout, MEMBER)
left  = [(s, v) for v in MY_VIDEOS for s in STAGE_LIST if not fresh.is_done(s, v)]

total = len(MY_VIDEOS) * len(STAGE_LIST)
print(f"{MEMBER}: {total - len(left)}/{total} việc đã lên {VOLUME}")
for stage in STAGE_LIST:
    n = sum(1 for v in MY_VIDEOS if fresh.is_done(stage, v))
    print(f"  {stage:<16} {n:>4}/{len(MY_VIDEOS)} video")

if left:
    print()
    print(f"còn {len(left)} việc — bấm Run All lại, nó chạy tiếp từ chỗ đứt")
else:
    print()
    print("XONG phần của bạn. Báo Lead.")
    print(f"  python scripts/verify_bundle.py --shards {SHARDS} --member {MEMBER}")